In [245]:
import os
import ast
import fitz
import torch
import base64
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from colpali_engine.models import ColQwen2, ColQwen2Processor

import psycopg2
from pgvector.psycopg2 import register_vector

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.messages import HumanMessage, SystemMessage

In [41]:
pdf_path = "pdfs\supplier_guide_detailed.pdf"

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_12772\394254503.py:1: SyntaxWarning: invalid escape sequence '\s'
  pdf_path = "pdfs\supplier_guide_detailed.pdf"


In [42]:
load_dotenv()

True

In [43]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=7,
    check_every_n_seconds=0.1,
    max_bucket_size=10
)

In [6]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, rate_limiter=rate_limiter)

In [7]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="rag_db",
    user="postgres"
)
register_vector(conn)
cur = conn.cursor()

In [8]:
model_name = "vidore/colqwen2-v1.0"
model = ColQwen2.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
).eval()


Loading weights: 100%|██████████| 392/392 [00:00<00:00, 5129.96it/s]
ColQwen2 LOAD REPORT from: vidore/colqwen2-v1.0
Key                                                                    | Status     | 
-----------------------------------------------------------------------+------------+-
base_model.language_model.model.custom_text_proj.lora_B.default.weight | UNEXPECTED | 
base_model.language_model.model.custom_text_proj.lora_A.default.weight | UNEXPECTED | 
custom_text_proj.lora_A.default.weight                                 | MISSING    | 
custom_text_proj.lora_B.default.weight                                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
processor = ColQwen2Processor.from_pretrained(
    model_name, 
    trust_remote_code=True
)

In [10]:
def get_coarse_embedding(embeddings_tensor):
    normalized = embeddings_tensor / embeddings_tensor.norm(dim=-1, keepdim=True)
    return normalized.mean(dim=1).squeeze().to(torch.float32).cpu() 

In [11]:
def embed_query(query, model, processor):
    inputs = processor.process_queries(queries=[query])
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        embeddings = model(**inputs)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        coarse_vector = get_coarse_embedding(embeddings).flatten().tolist()
        embeddings = embeddings.squeeze(0).to(torch.float32).cpu().numpy()
    return coarse_vector, embeddings


In [570]:
query = "If I am bidding for a project worth S$5,000, which procurement category applies?"

In [548]:
def numerical_expansion(query, llm):
    # Quick check: If no numbers/symbols, return original query to avoid 'drift'
    if not any(char.isdigit() or char == '$' for char in query):
        return query

    prompt = f"""
    Act as a search query optimizer for technical documents. 
    Rewrite the user query to maximize retrieval accuracy by following these rules:
    
    1. NUMERICAL BOUNDARIES: If a specific value or number is mentioned, include the 
       likely category or threshold it belongs to (e.g., if a user asks for 'age 5', 
       add 'early childhood' or 'pediatric').
    2. TECHNICAL SYNONYMS: Add common industry-standard synonyms for the actions described.
    3. NO CONVERSATION: Do not explain the change. Return only the keywords.
    
    User Query: {query}
    Optimized Query:"""
    
    return llm.invoke(prompt).content


In [571]:
query = numerical_expansion(query, llm)

In [572]:
print(query)

project bidding S$5,000 procurement category


In [573]:
coarse_vector, embeddings = embed_query(query, model, processor)

In [581]:
def hybrid_coarse_search(coarse_vector, query_text, conn, top_k=20):
    try:
        with conn.cursor() as cur:
            cur.execute('''
                WITH vector_results AS (
                    SELECT id, row_number() OVER (ORDER BY coarse_vector <=> %s::vector) as rank
                    FROM pages LIMIT 50
                ),
                keyword_results AS (
                    SELECT id, row_number() OVER (ORDER BY ts_rank(fts_tokens, websearch_to_tsquery('english', %s)) DESC) as rank
                    FROM pages 
                    WHERE fts_tokens @@ websearch_to_tsquery('english', %s) LIMIT 50
                )
                SELECT COALESCE(v.id, k.id) as id,
                       (COALESCE(1.0 / (60 + v.rank), 0.0) + 
                        COALESCE(1.0 / (60 + k.rank), 0.0)) as rrf_score
                FROM vector_results v
                FULL OUTER JOIN keyword_results k ON v.id = k.id
                ORDER BY rrf_score DESC LIMIT %s
            ''', (coarse_vector, query_text, query_text, top_k))
            return [row[0] for row in cur.fetchall()]
    except Exception as e:
        conn.rollback()
        return []

In [406]:
def coarse_search(coarse_vector, top_k = 20):
    try:
        with conn.cursor() as cur:
            cur.execute('''
                SELECT id FROM pages
                ORDER BY coarse_vector <=> %s::vector
                LIMIT %s

            ''', (coarse_vector, top_k))
            ids = [row[0] for row in cur.fetchall()]
            return ids
    except Exception as e:
        print(f"Error: {e}")
        conn.rollback()

In [574]:
ids = coarse_search(coarse_vector)

In [444]:
import numpy as np
import psycopg2

def precision_search_diverse(query_embeddings, ids, conn, top_k=10):
    """
    query_embeddings: (num_tokens, 128) - the ColQwen embeddings for your query
    ids: List of IDs from your coarse_search (Top 20)
    """
    try:
        # 1. BATCH FETCH: Get all patch embeddings for the candidate IDs in one go
        # This is much faster than looping 20 times
        all_page_data = {}
        with conn.cursor() as cur:
            cur.execute('''
                SELECT page_id, patch_embedding 
                FROM page_patches 
                WHERE page_id IN %s 
                ORDER BY page_id, patch_index
            ''', (tuple(ids),))
            
            rows = cur.fetchall()
            for p_id, emb in rows:
                if p_id not in all_page_data:
                    all_page_data[p_id] = []
                all_page_data[p_id].append(emb)

        # 2. CALCULATE MAXSIM SCORES for each page
        scored_pages = []
        for p_id, patches in all_page_data.items():
            page_matrix = np.array(patches, dtype=np.float32)
            
            # Matrix multiplication: (query_tokens, 128) @ (128, page_patches)
            # Resulting sim_matrix: (query_tokens, page_patches)
            sim_matrix = query_embeddings @ page_matrix.T
            
            # MaxSim: Best match score on this page for EACH token in your query
            token_scores = np.max(sim_matrix, axis=1) 
            scored_pages.append({
                "id": p_id,
                "token_scores": token_scores
            })

        # 3. DIVERSITY SELECTION (Marginal Gain Logic)
        # This ensures that if Page A covers 'Procurement', 
        # we hunt for Page B that covers 'Storage'.
        selected_pages = []
        # Track the best score we've found for each token so far
        best_scores_per_token = np.zeros(query_embeddings.shape[0])

        for _ in range(min(top_k, len(scored_pages))):
            best_marginal_gain = -1.0
            best_candidate = None
            
            for page in scored_pages:
                if page in selected_pages:
                    continue
                
                # Marginal Gain: How much NEW information does this page add
                # to the tokens we haven't matched well yet?
                # We subtract what we already know (best_scores_per_token) from current scores
                gains = np.maximum(0, page["token_scores"] - best_scores_per_token)
                marginal_gain = np.sum(gains)
                
                if marginal_gain > best_marginal_gain:
                    best_marginal_gain = marginal_gain
                    best_candidate = page
            
            if best_candidate:
                selected_pages.append(best_candidate)
                # Update our knowledge base with the strengths of the newly added page
                best_scores_per_token = np.maximum(best_scores_per_token, best_candidate["token_scores"])

        # Return just the IDs and scores for the generation step
        return [(p["id"], np.sum(p["token_scores"])) for p in selected_pages]

    except Exception as e:
        print(f"Database Error: {e}")
        conn.rollback()
        return []

# --- HOW TO USE IN YOUR LOOP ---
# coarse_ids = coarse_search(coarse_vector)
# results = precision_search_diverse(embeddings, coarse_ids, conn, top_k=3)


In [429]:
# MaxSim
def precision_search(embeddings, ids, top_k = 10):
    try: 
        result = []
        for p_id in ids:
            with conn.cursor() as cur:
                cur.execute('''
                    SELECT patch_embedding
                    FROM page_patches
                    WHERE page_id = %s
                    ORDER BY patch_index
                ''', (p_id,))
            
                rows = cur.fetchall()
                page_matrix = np.array([r[0] for r in rows], dtype = np.float32) # np matrix for page
                sim_matrix = embeddings @ page_matrix.T # matrix multiplication
                page_score = np.sum(np.max(sim_matrix, axis=1)) # sum best matching patches for page
                result.append((p_id, page_score))
        return sorted(result, key=lambda x: x[1], reverse=True)[:top_k]

    except Exception as e:
        print(f"Error: {e}")
        conn.rollback()

In [575]:
results = precision_search_diverse(embeddings, ids, conn)

In [409]:
def retrieve_content(results):
    output = []
    with conn.cursor() as cur:
        for id, score in results:
            cur.execute('''
                SELECT page_number, markdown
                FROM pages
                WHERE id = %s
            ''', (id,))

            row = cur.fetchone()
            output.append({
                "page_number": row[0],
                "markdown": row[1]
            })
    return output

In [576]:
content = retrieve_content(results)
content

[{'page_number': 6,
  'markdown': 'Based on the estimated value of the procurement, agencies would choose the procurement approach as shown in the following table:\n\n| Estimated Procurement Value (EPV)            | Procurement Approach                                                                    | Description                                                                                                                                                                                                                 | Sourcing Methods                                                                                    |\n|----------------------------------------------|-----------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|---------

In [179]:
def get_page_base64(pdf_path, page_number):
    doc = fitz.open(pdf_path)
    page = doc.load_page(page_number - 1)

    zoom = 1.0 
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat)

    img_bytes = pix.tobytes("jpg")
    return base64.b64encode(img_bytes).decode('utf-8')


In [180]:
def generate_context(content, pdf_path):
    context = []
    for dic in content:
        context.append({
            "type": "text",
            "text": f"--- Page {dic['page_number']} ---\n{dic['markdown']}"
        })

        b64_img = get_page_base64(pdf_path, dic["page_number"])

        context.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpg;base64,{b64_img}"}
        })
    return HumanMessage(content=context)

In [181]:
context = generate_context(content, pdf_path)

In [134]:
def generate_message(query, context):
    messages = [SystemMessage(content="""
    You are a professional assistant. Your goal is to provide accurate answers based ONLY on the provided context which includes the markdown and image of pages of a document.

    RULES:
    1. GROUNDING: If the answer is not contained within the provided context, state clearly that you do not have enough information. DO NOT USE EXTERNAL KNOWLEDGE.
    2. MARKDOWN: Use the markdown text for precise data extractions, quotes and tables.
    3. IMAGES: Use the images to verify layout, check for visual elements, and clarify ambiguous parts of the text.
    4. DISCREPANCY: Treat the page image and markdown as equal independent sources of information. ANY CLAIM OR INFORMATION MUST BE SUPPORTED BY BOTH SOURCES. Flag any discrepancies. 
    5. CITATIONS: You MUST CITE THE SPECIFIC PAGE NUMBER (e.g. [Page 4]) for every fact or claim you make. Each page is labelled at the start (e.g. --- Page 4 ---).
    6. CONTEXT INTEGRATION: Treat the context as a single unified knowledge base, even if they are provided out of chronological order.
    7. RELEVANCE: ONLY USE INFORMATION FROM THE CONTEXT that are relevant to answering the question.  
    8. FORMATTING: Use clear headings and bullet points for complex answers.                          
    """),
    context, 
    HumanMessage(content=f"""
    Question: {query}
    """)

    ]
    return messages

In [182]:
messages = generate_message(query, context)

In [183]:
for chunk in llm.stream(messages):
    print(chunk.content, end = "", flush = True)

For a procurement value of $150,000, the appropriate procurement approach is **Tender**. This falls under the category of tenders exceeding S$90,000, specifically:

- **Open Tender**: Tender notice published openly on GeBIZ inviting any interested supplier to bid according to the specified requirements.
- **Selective Tender**: A 2-stage process where interested suppliers are shortlisted based on their capabilities via an open pre-qualification exercise, and the shortlisted suppliers will be invited to submit their bids.
- **Limited Tender**: One or a few selected suppliers will be invited to tender.

This information is supported by the table on Page 6.

TESTING

In [604]:
query = "Procurement approach for large-scale tenders exceeding S$1 million and maximum file size."



In [605]:
query = numerical_expansion(query, llm)
coarse_vector, embeddings = embed_query(query, model, processor)
ids = coarse_search(coarse_vector)
results = precision_search_diverse(embeddings, ids, conn)
content = retrieve_content(results)
content

[{'page_number': 18,
  'markdown': "## 6. Bid Submission Checklist\n\nAll  information  submitted  as  part  of  the  bidding  process  must  be  completed  and  must  meet  the Government agencies' requirements as closely as possible. You may refer to the checklist below in your preparation of your proposal.\n\n1.\n\nHave you met all the critical criteria required by the agencies?\n\nYou  should  read  and  understand  the  criteria  before  preparing  your  proposal.\n\nProposals submitted to the agencies that meet the critical evaluation criteria will\n\nbe given the opportunity to be considered for award.\n\n2.\n\nHave you clarified all ambiguous points with the Government agency? If not, ask.\n\nYou should adopt a proactive approach to seek clarification with the agencies if\n\nyou do not fully understand the contractual terms of the Tender/Quotation.\n\n3.\n\nDoes your proposal meet the agencies' requirements? Is it coherent and easily\n\nunderstood by the agencies'?\n\nYour Tend